# Структура кода DAG

## Импорты


In [2]:
from datetime import datetime, timedelta

from airflow import DAG
from airflow.operators.bash import BashOperator
from airflow.utils.dates import days_ago


## Словарь аргументов

In [3]:
default_args = {
    'start_date': datetime(2020, 1, 1),
    'owner': 'airflow',
} 

## Объявление DAG

In [4]:
dag = DAG(
    "my_dag_name", 
    default_args=default_args,
    description = "Simple DAG",
    schedule_interval=None,
) 

/var/folders/fv/jb9qsjkd4_b_sx1z7304t_lr0000gn/T/ipykernel_71458/257905110.py:1 RemovedInAirflow3Warning: Param `schedule_interval` is deprecated and will be removed in a future release. Please use `schedule` instead.

## Объявление задач

In [5]:
# задача с Bash-командой, которая выводит дату в лог задачи
t1 = BashOperator(
    task_id='print_date',
    bash_command='date',
        dag=dag,
)

# задача с Bash-командой, которая делает задержку длиной 5 секунд
# в случае ошибки выполнения Airflow будет перезапускать задачу три раза 
t2 = BashOperator(
    task_id='sleep',
    bash_command='sleep 5',
    retries=3,
    dag=dag,
)

t1 >> t2 

<Task(BashOperator): sleep>

## DAG через контекстный менеджер

In [6]:
# секция импортов
from datetime import timedelta
from airflow import DAG
from airflow.operators.bash_operator import BashOperator
from airflow.utils.dates import days_ago


# словарь с параметрами по умолчанию для всех задач DAG
default_args = {
    'owner': 'airflow',                 # владелец DAG
    'depends_on_past': False,           # задача не зависит от предыдущего запуска
    'start_date': days_ago(2),          # DAG считается начатым 2 дня назад
    'email': ['airflow@example.com'],   # email для уведомлений
    'email_on_failure': False,          # не отправлять письмо при ошибке
    'email_on_retry': False,            # не отправлять письмо при повторе
    'retries': 1,                       # количество повторных запусков при ошибке
    'retry_delay': timedelta(minutes=5) # задержка между повторами
}

# контекстный менеджер DAG
# всё, что объявлено внутри блока with, автоматически относится к этому DAG
# созданный DAG сохраняется в переменную dag с помощью "as"
with DAG(
    dag_id='tutorial',                          # уникальное имя DAG
    default_args=default_args,                  # передаём параметры по умолчанию
    description='A simple tutorial DAG',        # описание DAG
    schedule_interval=timedelta(days=1),        # запуск один раз в день
) as dag:

    # первая задача: выводит дату в лог задачи
    t1 = BashOperator(
        task_id='print_date',   # уникальное имя задачи
        bash_command='date'     # Bash-команда, которую выполнит задача
    )


    # вторая задача: делает задержку длиной 5 секунд
    t2 = BashOperator(
        task_id='sleep',            # имя задачи
        depends_on_past=False,      # не зависит от предыдущего запуска
        bash_command='sleep 5',     # команда ожидания
        retries=3                   # количество повторов при ошибке
    )


    # порядок выполнения задач: сначала выполняется t1, затем t2
    t1 >> t2

/var/folders/fv/jb9qsjkd4_b_sx1z7304t_lr0000gn/T/ipykernel_71458/3594302547.py:4 DeprecationWarning: The `airflow.operators.bash_operator.BashOperator` class is deprecated. Please use `'airflow.operators.bash.BashOperator'`.

/var/folders/fv/jb9qsjkd4_b_sx1z7304t_lr0000gn/T/ipykernel_71458/3594302547.py:12 RemovedInAirflow3Warning: Function `days_ago` is deprecated and will be removed in Airflow 3.0. You can achieve equivalent behavior with `pendulum.today('UTC').add(days=-N, ...)`

/var/folders/fv/jb9qsjkd4_b_sx1z7304t_lr0000gn/T/ipykernel_71458/3594302547.py:23 RemovedInAirflow3Warning: Param `schedule_interval` is deprecated and will be removed in a future release. Please use `schedule` instead.

In [7]:
# 1. Импортируем библиотеки
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.bash import BashOperator
from datetime import datetime, timedelta

# 2. Объявляем словарь аргументов
default_args = {
    'start_date': datetime(2026, 1, 1),
    'retries': 1,
    'retry_delay': timedelta(minutes=5)
}

# 3. Объявляем функции
# функция рассчитывает дату неделю назад от актуального числа
def calculate_time_window():
    today = datetime.today()
    window_days = 7

    window_start = today - timedelta(days=window_days)
    window_end = today

    print(f"Начало окна: {window_start.date()}")
    print(f"Конец окна: {window_end.date()}")
    print(f"Размер окна: {window_days} дней")

# 4. Объявление DAG
dag = DAG(
    dag_id="calculate_time_window_dag",
    default_args=default_args,
    schedule_interval="@daily",
    catchup=False,
    description="Печатает дату выполнения и рассчитывает временное окно",
)

# 5. Объявление задач
print_date_task = BashOperator(
    task_id="print_execution_date",
    bash_command="date '+%Y-%m-%d'",
    dag=dag,
)
calculate_window_task = PythonOperator(
    task_id="calculate_time_window",
    python_callable=calculate_time_window,
    dag=dag,
)

# 6. Указываем последовательность выполнения задач
print_date_task >> calculate_window_task

/var/folders/fv/jb9qsjkd4_b_sx1z7304t_lr0000gn/T/ipykernel_71458/901344251.py:28 RemovedInAirflow3Warning: Param `schedule_interval` is deprecated and will be removed in a future release. Please use `schedule` instead.

/Users/arseniy/study/practicum/data_science/practicum_ds/.venv/lib/python3.9/site-packages/cron_descriptor/ExpressionDescriptor.py:69 UserWarning: No system locale set. Falling back to 'en_US'. Set LANG/LC_ALL or pass locale_code to override.

<Task(PythonOperator): calculate_time_window>